## Modèles récurrents : apprendre l'allocation de Markowitz

1. **la cible** : ce que le modèle doit reproduire ;
2. **les trois architectures** V0, V1 et V2 ;
3. **les fonctions de perte** : entropie croisée et Sharpe ;
4. **le chargement des données**, la construction des séquences et la normalisation ;
5. **le protocole d'évaluation** : sélection de modèle, évaluation finale, références ;
6. **V0**, 7. **V1** puis **V2** : validation croisée et choix de la configuration ;
8. **l'évaluation finale** sur le test ;
9. **annexe** : premiers essais sur le découpage simple.

## 1. La cible : l'oracle de Markowitz

**Pour chaque date $t$, l'oracle est le portefeuille long-only qui a maximisé le Sharpe sur les 63 jours suivants (de $t+1$ à $t+63$).** Il vaut **100 % CASH** dans les rares cas (2,4 % des dates) où aucun portefeuille d'actions n'a battu le taux sans risque.

Le modèle doit reproduire cette allocation en n'utilisant que l'information disponible à $t$.

## 2. Les trois architectures

Tous les modèles prennent une décision à la date $t$ et sortent **16 poids** (15 actions + CASH) via un **softmax** : les poids sont positifs et somment à 1, exactement comme $Y_t$.

|  | **V0** | **V1** | **V2** |
|---|---|---|---|
| Ce que lit le LSTM | 1 séquence de 60 jours × 15 actions | 15 séquences de 52 semaines × 2 (action, CAC) | idem V1 |
| Tenseur d'entrée du LSTM | (B, 60, 15) | (B·15, 52, 2) | idem V1 |
| Poids du LSTM | un seul LSTM pour tout | **un seul LSTM partagé** par les 15 actions | idem V1 |
| Sortie du LSTM | 1 état pour l'ensemble du marché | 15 états, un par action | idem V1 |
| Ajouté avant la tête de sortie | rien | rien | 10 features de l'action + 10 features globales |
| Score du CASH | depuis l'état unique | depuis la moyenne des 15 états | depuis les features globales |
| Sortie | 16 poids (softmax) | 16 poids (softmax) | 16 poids (softmax) |

B = taille du batch (nombre de dates traitées ensemble).

### Points communs aux trois versions

- **Fenêtre fixe** : on donne toujours la même longueur (60 jours ou 52 semaines), à l'entraînement comme en inférence. Pour une autre longueur, on réentraîne et on compare sur la validation.
- **Plusieurs entrées → une sortie** : le LSTM lit toute la fenêtre (état initial à 0), et **seul l'état final** produit l'allocation. Il n'y a ni prédiction intermédiaire ni autorégression : le LSTM sert d'**encodeur** qui résume le passé.
- **Taille de l'état caché** $h$ (8, 16 ou 32) : un hyperparamètre, choisi sur la validation. À ne pas confondre avec la longueur de la séquence.

## 3. Les fonctions de perte

Notations pour un échantillon (une date $t$) :
- $\pi \in \mathbb{R}^{16}$ : poids prédits (sortie du softmax) ;
- $Y \in \mathbb{R}^{16}$ : poids de l'oracle ;
- $R \in \mathbb{R}^{16}$ : rendements réalisés de chaque actif entre $t+1$ et $t+63$ (`R_fwd`), et $R_{\text{cash}}$ celui du CASH.

### 3.1 Entropie croisée

$$
\mathcal{L}_{CE} = -\sum_{i=1}^{16} Y_i \log \pi_i
$$

### 3.2 Sharpe : optimiser directement la performance

Pour un batch de $B$ dates, le rendement en excès du portefeuille prédit est $r_b = \pi_b^\top R_b - R_{\text{cash},b}$, et :

$$
\mathcal{L}_{Sharpe} = -\,\frac{\operatorname{moyenne}_b(r_b)}{\operatorname{écart\text{-}type}_b(r_b)}
$$

- Cette perte récompense directement les allocations qui ont bien performé.
- Le Sharpe est estimé sur le batch : il faut des **batches d'au moins 128 dates**, sinon l'estimation est trop bruitée.
- Elle permet d'exploiter la partie **prévisible** du problème : le risque (la volatilité passée prédit bien la volatilité future), alors que les rendements futurs sont quasiment imprévisibles.

### 3.3 Hybride

$$
\mathcal{L} = \mathcal{L}_{CE} + \alpha \, \mathcal{L}_{Sharpe}, \qquad \alpha \in \{0 ;\, 0{,}1 ;\, 0{,}5\}
$$

$\alpha = 0$ : imitation pure. $\alpha > 0$ : imiter l'oracle tout en étant récompensé par la performance réalisée.

### 3.4 Critères de réussite

- Entropie croisée de validation inférieure à celle de la stratégie « toujours prédire la moyenne des cibles du train ». Sinon, le modèle n'a rien appris au-delà de la répartition moyenne.
- **Sharpe au moins comparable à celui du portefeuille équipondéré**, idéalement avec moins de turnover.

### Chargement des données

Fichiers produits par `data_generation/build_dataset.py`. Le découpage est déjà fait dans `meta.csv` : train 2006–2016, val 2017–2019, test 2020–2025 ; les dates `none` sont purgées (leur cible chevauche le bloc suivant).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA, N = Path("data/processed/N63_wmax1"), 63
read = lambda f: pd.read_csv(f, index_col=0, parse_dates=True)
X, Y, R_fwd, meta, rets = (read(DATA / f"{f}.csv") for f in ["X", "Y", "R_fwd", "meta", "returns"])
TICKERS = list(Y.columns[:-1])

idx = {s: np.flatnonzero(meta["split"].to_numpy() == s) for s in ["train", "val", "test"]}
for s, i in idx.items():
    print(f"{s:5s} : {len(i):4d} dates ({X.index[i[0]].date()} → {X.index[i[-1]].date()}), ~{len(i) // N} trimestres indépendants")

train : 2749 dates (2006-01-02 → 2016-10-03), ~43 trimestres indépendants
val   :  703 dates (2017-01-02 → 2019-10-02), ~11 trimestres indépendants
test  : 1474 dates (2020-01-02 → 2025-09-30), ~23 trimestres indépendants


Construction des entrées (rendements en log) :
- **V0** : les 60 derniers rendements journaliers des 15 actions → (dates, 60, 15) ;
- **V1/V2** : pour chaque action, 52 rendements hebdomadaires, avec ceux du CAC en 2ᵉ canal → (dates, 15, 52, 2). Le CAC est recalculé depuis `data/raw/prices.csv` ;
- **V2** : features par action (dates, 15, 10) et globales (dates, 10).

In [17]:
logr = np.log1p(rets.to_numpy())
pos = rets.index.get_indexer(X.index)                  # position de chaque date t dans le calendrier

# V0
seq_v0 = np.stack([logr[i - 59:i + 1] for i in pos])

# V1/V2 : CAC avec le même nettoyage que build_dataset.py
px = read("data/raw/prices.csv").dropna(how="all").ffill(limit=2).dropna(how="any")
cac = px["^FCHI"].pct_change().iloc[1:]
assert cac.index.equals(rets.index)
log_cac = np.log1p(cac.to_numpy())

def weekly(lr, n_weeks=52, step=5):
    # rendements log hebdomadaires, la dernière semaine finissant à t (semaines manquantes = 0)
    C = np.concatenate([np.zeros((1,) + lr.shape[1:]), lr.cumsum(axis=0)])
    out = np.zeros((len(pos), n_weeks) + lr.shape[1:])
    for k, i in enumerate(pos):
        ends = i + 1 - step * np.arange(n_weeks)[::-1]
        ok = ends >= step
        out[k, ok] = C[ends[ok]] - C[ends[ok] - step]
    return out

seq_v1 = np.stack([weekly(logr).transpose(0, 2, 1),
                   np.repeat(weekly(log_cac)[:, None, :], 15, axis=1)], axis=-1)

# V2 : features rangées par action, et globales
ASSET_FEATS = ["mom_5", "mom_21", "mom_63", "mom_126", "mom_252", "vol_21", "vol_63", "dvol_63", "dd_252", "corrmkt_63"]
feat_asset = np.stack([X[[f"{f}__{tk}" for f in ASSET_FEATS]].to_numpy() for tk in TICKERS], axis=1)
feat_glob = X[[c for c in X.columns if c.endswith("__global") or c.startswith(("market__", "macro__"))]].to_numpy()

print("seq_v0", seq_v0.shape, "| seq_v1", seq_v1.shape, "| feat_asset", feat_asset.shape, "| feat_glob", feat_glob.shape)

seq_v0 (5082, 60, 15) | seq_v1 (5082, 15, 52, 2) | feat_asset (5082, 15, 10) | feat_glob (5082, 10)


### Normalisation
Statistiques calculées **sur le train uniquement**. Les rendements des actions partagent **une seule échelle** (normaliser action par action effacerait leurs différences de volatilité, qui sont informatives). Les features par action sont normalisées par type de feature, les globales par colonne. $Y$ et `R_fwd` ne sont pas normalisés.

In [3]:
tr = idx["train"]
seq_v0_n = seq_v0 / seq_v0[tr].std()
seq_v1_n = seq_v1 / np.array([seq_v1[tr][..., 0].std(), seq_v1[tr][..., 1].std()])
feat_asset_n = (feat_asset - feat_asset[tr].mean(axis=(0, 1))) / feat_asset[tr].std(axis=(0, 1))
feat_glob_n = (feat_glob - feat_glob[tr].mean(axis=0)) / feat_glob[tr].std(axis=0)

for name, a in [("seq_v0", seq_v0_n), ("seq_v1", seq_v1_n), ("feat_asset", feat_asset_n), ("feat_glob", feat_glob_n)]:
    print(f"{name:10s} écart-type  train {a[tr].std():.2f} | val {a[idx['val']].std():.2f} | test {a[idx['test']].std():.2f}")

seq_v0     écart-type  train 1.00 | val 0.59 | test 0.92
seq_v1     écart-type  train 1.00 | val 0.64 | test 0.96
feat_asset écart-type  train 1.00 | val 0.77 | test 1.07
feat_glob  écart-type  train 1.00 | val 1.20 | test 1.52


## 5. Protocole d'évaluation

Ce protocole est fixé **avant** les expériences et s'applique à tous les modèles (V0, V1, …) et à toutes les références.

**Étape A : sélection de modèle (2006–2019)**
- Validation croisée **walk-forward** sur 4 plis : on s'entraîne sur tout le passé, on valide sur les 2 années suivantes (2012–13, 2014–15, 2016–17, 2018–19). Purge : un échantillon d'entraînement n'est gardé que si sa cible finit avant le début de la validation. Normalisation recalculée sur le train de chaque pli.
- Entraînement : entropie croisée, Adam, arrêt anticipé sur l'entropie croisée de validation du pli.
- **Critère de sélection : l'entropie croisée de validation, moyenne sur les 4 plis** (c'est la perte d'entraînement, et la métrique la moins bruitée).
- **Règle de choix** : parmi les configurations à moins d'une erreur standard de la meilleure (écart-type sur les plis / √4), on retient la plus simple (le moins de paramètres, puis le plus de dropout). Le Sharpe de validation est affiché, mais ne sert pas au choix.

**Étape B : évaluation finale (2020–2025, une seule fois)**
- La configuration retenue est réentraînée sur 2006–2019 (purgé avant 2020) pendant le nombre d'époques médian trouvé en validation croisée, avec 5 graines.
- **Métriques**, identiques pour les modèles et les références :
  - entropie croisée (distance à l'oracle) ;
  - **Sharpe** en excès du CASH, rééquilibrage tous les 63 jours, moyenne (± écart-type) sur les 63 dates de départ : **la métrique principale** ;
  - turnover moyen entre deux rééquilibrages (somme des |Δpoids|, sans dérive des prix) ;
  - nombre effectif d'actifs, 1 / Σ poids².

**Références**, calculées avec les mêmes métriques sur les mêmes périodes :
- **oracle** : plafond inatteignable (son entropie croisée est le plancher théorique) ;
- **moyenne des cibles du train** : le seuil d'entropie croisée ; un modèle au-dessus n'a rien appris au-delà de la répartition moyenne ;
- **équipondéré** et **inverse de la volatilité** (poids ∝ 1 / `vol_63`) : les seuils de Sharpe à battre.

In [4]:
# --- Périodes ---
label_end = pd.to_datetime(meta["label_end"])
def purged_train(before):          # dates d'entraînement depuis 2006 dont la cible finit avant `before`
    return np.flatnonzero((X.index >= "2006-01-01") & (label_end < pd.Timestamp(before)).to_numpy())

FOLDS = {}
for y0 in [2012, 2014, 2016, 2018]:
    va = np.flatnonzero((X.index >= f"{y0}-01-01") & (label_end < pd.Timestamp(f"{y0 + 2}-01-01")).to_numpy())
    FOLDS[f"{y0}-{y0 + 1}"] = (purged_train(f"{y0}-01-01"), va)
FINAL = (purged_train("2020-01-01"), idx["test"])           # étape B : réentraînement final, puis test

# --- Métriques communes ---
Yn, Rn = Y.to_numpy(), R_fwd.to_numpy()
def metrics(W, i):
    # W : poids (len(i), 16) prédits aux dates i
    sharpe, turnover = [], []
    for o in range(N):                                       # rééquilibrage tous les N jours, N dates de départ
        w, r = W[o::N], Rn[i][o::N]
        ex = (w * r).sum(1) - r[:, -1]
        sharpe.append(ex.mean() / ex.std() * np.sqrt(252 / N))
        turnover.append(np.abs(np.diff(w, axis=0)).sum(1).mean())
    return {"ce": -(Yn[i] * np.log(W + 1e-8)).sum(1).mean(), "sharpe": np.mean(sharpe),
            "sharpe_std": np.std(sharpe), "turnover": np.mean(turnover), "n_eff": (1 / (W ** 2).sum(1)).mean()}

# --- Références ---
inv_vol = 1 / X[[f"vol_63__{t}" for t in TICKERS]].to_numpy()
def baselines(tr, i):
    n = len(i)
    return {"Oracle": Yn[i],
            "Moyenne des cibles du train": np.tile(Yn[tr].mean(0), (n, 1)),
            "Équipondéré": np.c_[np.full((n, 15), 1 / 15), np.zeros(n)],
            "Inverse volatilité": np.c_[inv_vol[i] / inv_vol[i].sum(1, keepdims=True), np.zeros(n)]}

# --- Règle de choix ---
def select(summary):
    best = summary.iloc[0]
    ok = summary[summary["val_loss"] <= best["val_loss"] + best["loss_std"] / np.sqrt(len(FOLDS))]
    return ok.sort_values(["n_params", "dropout"], ascending=[True, False]).iloc[0]

ref = pd.DataFrame([{"période": p, "stratégie": s, **metrics(W, i)}
                    for p, (tr, i) in {**FOLDS, "test": FINAL}.items() for s, W in baselines(tr, i).items()])
ref_ce = ref[ref["stratégie"] == "Moyenne des cibles du train"].set_index("période")["ce"].to_dict()
COLS = ["ce", "sharpe", "sharpe_std", "turnover", "n_eff"]
print("Références : moyenne sur les 4 plis (étape A)")
display(ref[ref["période"] != "test"].groupby("stratégie", sort=False)[COLS].mean().round(3))
print("Références : test 2020–2025 (étape B)")
ref[ref["période"] == "test"].set_index("stratégie")[COLS].round(3)

Références : moyenne sur les 4 plis (étape A)


,ce,sharpe,sharpe_std,turnover,n_eff
stratégie,,,,,
Oracle,1.044,4.806,1.234,1.707,2.904
Moyenne des cibles du train,2.721,1.508,0.445,0.000,13.446
Équipondéré,2.726,1.475,0.413,0.000,15.000
Inverse volatilité,2.699,1.539,0.465,0.131,14.265


Références : test 2020–2025 (étape B)


,ce,sharpe,sharpe_std,turnover,n_eff
stratégie,,,,,
Oracle,1.225,4.931,0.835,1.593,3.371
Moyenne des cibles du train,2.791,0.775,0.282,0.000,13.419
Équipondéré,2.921,0.845,0.280,0.000,15.000
Inverse volatilité,2.925,0.804,0.284,0.154,13.908


## 6. V0 : LSTM / GRU global sur les rendements journaliers

Grille testée : type de cellule, taille de l'état caché, nombre de couches, dropout (profondeur fixée à 60 jours). `grid_search` entraîne chaque configuration sur un couple (train, validation) ; `cross_validate` l'applique aux 4 plis, résume les résultats et applique la règle de choix du protocole.

In [5]:
import itertools, time
import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", DEVICE)

device : cpu


In [6]:
# Architecture V0
class V0(nn.Module):
    def __init__(self, model="lstm", hidden=16, layers=1, dropout=0.0, n_assets=15):
        super().__init__()
        rnn = nn.LSTM if model == "lstm" else nn.GRU
        self.rec = rnn(
            input_size = n_assets,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, n_assets +1)

    def forward(self,x):
        out, _ =self.rec(x)
        last = out[:,-1]
        return torch.softmax(self.head(self.drop(last)), dim=-1)   # (B, 16), log des poids

In [21]:
def grid_search(grid, tr_i=None, va_i=None, seed=0, lr=1e-3, weight_decay=1e-4, batch=128, max_epochs=100, patience=5, cross_entropy=True, arch="v0", fixed_epochs=None):
    # Teste toutes les combinaisons de `grid` : entraînement sur tr_i, arrêt anticipé et score sur va_i
    # (par défaut : le train et la validation du découpage de base).
    tr_i = idx["train"] if tr_i is None else tr_i
    va_i = idx["val"] if va_i is None else va_i
    Yt = torch.tensor(Y.to_numpy(), dtype=torch.float32)
    if cross_entropy:
        criterion = lambda p, y: -(y * torch.log(p + 1e-8)).sum(-1).mean()

    combos = list(itertools.product(*grid.values()))
    results = []
    for n, values in enumerate(combos, 1):
        cfg = dict(zip(grid.keys(), values))
        t0 = time.time()

        if arch == "v0":   # séquences journalières (dates, L, 15), normalisées avec l'écart-type du train
            L = cfg.get("L", 60)
            S = np.stack([logr[i - L + 1:i + 1] for i in pos]).astype(np.float32)
            S /= S[tr_i].std()
            inputs = [S]
        else:              # V1/V2 : séquences hebdomadaires (dates, 15, 52, 2), un écart-type par canal
            S = seq_v1.astype(np.float32)
            S /= np.array([S[tr_i][..., 0].std(), S[tr_i][..., 1].std()], dtype=np.float32)
            inputs = [S]
            if arch == "v2":   # + features par action (par type de feature) et globales (par colonne)
                FA = feat_asset.astype(np.float32)
                FA = (FA - FA[tr_i].mean(axis=(0, 1))) / FA[tr_i].std(axis=(0, 1))
                FG = feat_glob.astype(np.float32)
                FG = (FG - FG[tr_i].mean(axis=0)) / FG[tr_i].std(axis=0)
                inputs += [FA, FG]
        x_tr = [torch.tensor(a[tr_i]) for a in inputs]
        x_va = [torch.tensor(a[va_i]).to(DEVICE) for a in inputs]
        y_tr, y_va = Yt[tr_i], Yt[va_i].to(DEVICE)

        torch.manual_seed(seed)
        model = {"v0": V0, "v1": V1, "v2": V2}[arch](cfg["model"], cfg["hidden"], cfg["layers"], cfg["dropout"]).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best, best_ep, best_state, wait = float("inf"), 0, None, 0
        for ep in range(fixed_epochs or max_epochs):
            model.train()
            perm = torch.randperm(len(tr_i))                      # mélange à l'intérieur du train
            for b in range(0, len(perm), batch):
                j = perm[b:b + batch]
                loss = criterion(model(*[x[j].to(DEVICE) for x in x_tr]), y_tr[j].to(DEVICE))
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(*x_va), y_va).item()
            if fixed_epochs:                                      # réentraînement final : nombre d'époques fixé
                best, best_ep, best_state = val_loss, ep + 1, {k: t.clone() for k, t in model.state_dict().items()}
                continue
            if val_loss < best - 1e-4:                            # arrêt anticipé
                best, best_ep, wait = val_loss, ep + 1, 0
                best_state = {k: t.clone() for k, t in model.state_dict().items()}
            else:
                wait += 1
                if wait >= patience:
                    break
        model.load_state_dict(best_state)

        # métriques communes du protocole sur va_i
        with torch.no_grad():
            m = metrics(model(*x_va).cpu().numpy(), va_i)

        results.append({**cfg, "val_loss": best, **{f"val_{k}": v for k, v in m.items()}, "best_epoch": best_ep,
                        "n_params": sum(p.numel() for p in model.parameters()), "sec": round(time.time() - t0, 1)})
        print(f"[{n}/{len(combos)}] {cfg} → loss {best:.3f} | Sharpe {m['sharpe']:+.2f} "
              f"| {best_ep} époques | {time.time() - t0:.0f} s")

    return pd.DataFrame(results).sort_values("val_loss").reset_index(drop=True)

In [22]:
def cross_validate(grid, arch):
    # étape A du protocole : grid search sur les 4 plis, résumé par configuration, règle de choix
    res = pd.concat([grid_search(grid, tr, va, arch=arch).assign(fold=name) for name, (tr, va) in FOLDS.items()])
    res["gain_vs_ref"] = res["fold"].map(ref_ce) - res["val_loss"]          # > 0 : mieux que la moyenne des cibles
    res.to_csv(f"models/lstm/results/cv_{arch}_seed0.csv", index=False)
    summary = (res.groupby(list(grid))
               .agg(val_loss=("val_loss", "mean"), loss_std=("val_loss", "std"), gain_vs_ref=("gain_vs_ref", "mean"),
                    val_sharpe=("val_sharpe", "mean"), epochs=("best_epoch", "median"), n_params=("n_params", "first"))
               .sort_values("val_loss").reset_index())
    best = select(summary)
    top = summary.iloc[0]
    print(f"meilleure perte moyenne : {top[list(grid)].to_dict()} | {int(top['epochs'])} époques")
    print(f"règle de l'écart-type   : {best[list(grid)].to_dict()} | {int(best['epochs'])} époques")
    return summary, best

GRID = {"model": ["lstm", "gru"], "hidden": [8, 16, 32], "layers": [1, 2], "dropout": [0.0, 0.2]}

In [9]:
summary_v0, best_v0 = cross_validate(GRID, "v0")
summary_v0.round(3)

[1/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0} → loss 2.697 | Sharpe +1.84 | 8 époques | 5 s
[2/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2} → loss 2.699 | Sharpe +1.84 | 8 époques | 1 s
[3/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0} → loss 2.755 | Sharpe +1.73 | 1 époques | 1 s
[4/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.2} → loss 2.755 | Sharpe +1.73 | 1 époques | 1 s
[5/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.0} → loss 2.780 | Sharpe +1.70 | 6 époques | 1 s
[6/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.2} → loss 2.780 | Sharpe +1.70 | 6 époques | 1 s
[7/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.730 | Sharpe +1.79 | 1 époques | 1 s
[8/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.729 | Sharpe +1.79 | 1 époques | 1 s
[9/24] {'model': 'lstm', 'hidden': 32, 'layers': 1, 'dropout': 0.0} → loss 2.773 | Sharpe +1

,model,hidden,layers,dropout,val_loss,loss_std,gain_vs_ref,val_sharpe,epochs,n_params
0,lstm,16,2,0.0,2.681,0.080,0.040,1.624,2.0,4560
1,lstm,16,2,0.2,2.686,0.074,0.036,1.636,2.5,4560
2,lstm,8,1,0.2,2.686,0.063,0.035,1.598,9.0,944
3,lstm,8,1,0.0,2.687,0.061,0.034,1.593,7.5,944
4,lstm,32,1,0.0,2.699,0.077,0.022,1.583,3.0,6800
5,lstm,32,1,0.2,2.700,0.076,0.021,1.576,3.5,6800
6,gru,8,1,0.0,2.700,0.067,0.021,1.551,6.0,744
7,gru,8,1,0.2,2.702,0.065,0.019,1.547,7.0,744
8,lstm,16,1,0.2,2.704,0.078,0.017,1.566,6.0,2384
9,lstm,32,2,0.0,2.704,0.080,0.017,1.575,2.0,15248


Les modeles apprennent a peine plus que plus que la répartition moyenne des cibles, mais très peu, et le choix des hyperparamètres ne compte presque pas. On pourra testerr de maximiser le sharpe pour voir si cela fait mieux.  
On retient : 
### LSTM : taille état caché 8, 1 couche

## 7. V1 : LSTM / GRU partagé par action, rendements hebdomadaires

Même grille, mêmes plis, même règle de choix que V0 ; seule l'architecture change. Pour conclure, V1 doit battre la moyenne des cibles en entropie croisée (gain positif) et, en Sharpe, l'inverse de la volatilité, pas seulement l'équipondéré.

In [23]:
class V1(nn.Module):
    def __init__(self, model="lstm", hidden=8, layers=1, dropout=0.0, n_channels=2):
        super().__init__()
        rnn = nn.LSTM if model == "lstm" else nn.GRU
        self.rec = rnn(
            input_size=n_channels,              # 2 canaux : rendement de l'action, rendement du CAC
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.score = nn.Linear(hidden, 1)       # même tête pour toutes les actions : état -> score
        self.cash = nn.Linear(hidden, 1)        # score du CASH, à partir de la moyenne des états

    def forward(self, x):                       # x : (B, 15, 52, 2)
        B, n_assets, T, C = x.shape
        out, _ = self.rec(x.reshape(B * n_assets, T, C))     # les 15 séquences passent dans le même réseau
        h = self.drop(out[:, -1]).reshape(B, n_assets, -1)   # état final de chaque action : (B, 15, hidden)
        s_assets = self.score(h).squeeze(-1)                 # (B, 15)
        s_cash = self.cash(h.mean(dim=1))                    # (B, 1)
        return torch.softmax(torch.cat([s_assets, s_cash], dim=-1), dim=-1)   # (B, 16)

In [11]:
summary_v1, best_v1 = cross_validate(GRID, "v1")
summary_v1.round(3)

[1/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0} → loss 2.765 | Sharpe +1.79 | 10 époques | 16 s
[2/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2} → loss 2.767 | Sharpe +1.78 | 11 époques | 17 s
[3/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0} → loss 2.674 | Sharpe +1.83 | 26 époques | 108 s
[4/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.2} → loss 2.765 | Sharpe +1.75 | 9 époques | 39 s
[5/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.0} → loss 2.753 | Sharpe +1.74 | 1 époques | 12 s
[6/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.2} → loss 2.753 | Sharpe +1.74 | 1 époques | 12 s
[7/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.738 | Sharpe +1.87 | 8 époques | 87 s
[8/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.737 | Sharpe +1.84 | 10 époques | 53 s
[9/24] {'model': 'lstm', 'hidden': 32, 'layers': 1, 'dropout': 0.0} → loss 2.76

,model,hidden,layers,dropout,val_loss,loss_std,gain_vs_ref,val_sharpe,epochs,n_params
0,lstm,16,2,0.0,2.716,0.043,0.005,1.509,7.0,3490
1,gru,8,2,0.2,2.719,0.053,0.002,1.508,5.0,738
2,lstm,8,2,0.0,2.721,0.033,-0.000,1.500,9.5,978
3,lstm,16,2,0.2,2.725,0.030,-0.004,1.489,8.5,3490
4,gru,8,2,0.0,2.727,0.039,-0.006,1.472,1.5,738
5,gru,8,1,0.2,2.730,0.021,-0.009,1.502,15.0,306
6,lstm,16,1,0.0,2.732,0.026,-0.011,1.477,5.5,1314
7,gru,8,1,0.0,2.734,0.020,-0.012,1.497,10.5,306
8,gru,16,1,0.0,2.735,0.028,-0.014,1.452,7.0,994
9,lstm,16,1,0.2,2.736,0.018,-0.015,1.479,6.0,1314


## 7 bis. V2 : V1 + features

Même lecture des séquences que V1 (un LSTM partagé par les 15 actions). Avant la tête, on concatène à l'état de chaque action **ses 10 features** et les **10 features globales** ; le score du CASH vient des features globales. La tête reste partagée : (h + 20) poids, au lieu de 2 700 pour la tête de V0F.

**Grille réduite** : la partie séquence reprend la configuration retenue pour V1 (LSTM, h = 16, 2 couches) ; seul le dropout varie, {0 ; 0,2}. Mêmes plis, même perte, même règle de choix que pour V0 et V1 (meilleure perte moyenne en validation croisée).

**Avant de lancer cette section**, réexécuter les cellules `grid_search` et `cross_validate` de la section 6 : elles ont été modifiées pour accepter `arch="v2"`. La cellule ci-dessous vérifie que c'est bien fait.

In [24]:
class V2(nn.Module):
    def __init__(self, model="lstm", hidden=16, layers=2, dropout=0.0, n_channels=2, n_feat_asset=10, n_feat_glob=10):
        super().__init__()
        rnn = nn.LSTM if model == "lstm" else nn.GRU
        self.rec = rnn(n_channels, hidden, num_layers=layers, batch_first=True,
                       dropout=dropout if layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.score = nn.Linear(hidden + n_feat_asset + n_feat_glob, 1)   # même tête pour toutes les actions
        self.cash = nn.Linear(n_feat_glob, 1)                             # score du CASH : features globales

    def forward(self, x, fa, fg):              # x : (B, 15, 52, 2)  fa : (B, 15, 10)  fg : (B, 10)
        B, n_assets, T, C = x.shape
        out, _ = self.rec(x.reshape(B * n_assets, T, C))
        h = out[:, -1].reshape(B, n_assets, -1)                          # état final de chaque action
        z = torch.cat([h, fa, fg.unsqueeze(1).expand(-1, n_assets, -1)], dim=-1)
        s_assets = self.score(self.drop(z)).squeeze(-1)                  # (B, 15)
        s_cash = self.cash(fg)                                           # (B, 1)
        return torch.softmax(torch.cat([s_assets, s_cash], dim=-1), dim=-1)

In [25]:
import inspect
assert 'arch == "v2"' in inspect.getsource(grid_search), \
    "grid_search est l'ancienne version : réexécuter d'abord les cellules grid_search et cross_validate (section 6)"

GRID_V2 = {"model": ["lstm"], "hidden": [16], "layers": [2], "dropout": [0.0, 0.2]}
summary_v2, _ = cross_validate(GRID_V2, "v2")
summary_v2.round(3)

[1/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.704 | Sharpe +1.99 | 18 époques | 31 s
[2/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.689 | Sharpe +2.03 | 20 époques | 40 s
[1/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.758 | Sharpe +1.07 | 5 époques | 18 s
[2/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.763 | Sharpe +1.08 | 9 époques | 29 s
[1/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.738 | Sharpe +2.21 | 4 époques | 25 s
[2/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.741 | Sharpe +2.21 | 4 époques | 24 s
[1/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.692 | Sharpe +0.95 | 15 époques | 58 s
[2/2] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.701 | Sharpe +0.89 | 17 époques | 70 s
meilleure perte moyenne : {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.

,model,hidden,layers,dropout,val_loss,loss_std,gain_vs_ref,val_sharpe,epochs,n_params
0,lstm,16,2,0.2,2.723,0.034,-0.002,1.551,13.0,3504
1,lstm,16,2,0.0,2.723,0.030,-0.002,1.553,10.0,3504


## 8. Évaluation finale sur le test (étape B, une seule fois)

Chaque configuration retenue est réentraînée sur 2006–2019 pendant son nombre d'époques médian, avec 5 graines ; on rapporte la moyenne et l'écart-type des métriques sur les graines, à comparer aux références du test (section 5). À ne lancer qu'une fois les choix arrêtés : `RUN_TEST = True`.

In [20]:
RUN_TEST = False

if RUN_TEST:
    finals = []
    for arch, best in [("v0", best_v0), ("v1", best_v1)]:
        cfg = {k: [type(GRID[k][0])(best[k])] for k in GRID}                # une seule configuration
        for seed in range(5):
            finals.append(grid_search(cfg, *FINAL, seed=seed, arch=arch, fixed_epochs=int(best["epochs"]))
                          .assign(modèle=arch.upper()))
    finals = pd.concat(finals)
    display(finals.groupby("modèle")[[f"val_{c}" for c in COLS]].agg(["mean", "std"]).round(3))
    display(ref[ref["période"] == "test"].set_index("stratégie")[COLS].round(3))

**Choix figés avant le test.** Règle retenue : la **meilleure perte moyenne** en validation croisée (ligne 0 de chaque résumé), appliquée de la même façon à V0 et V1 :
- V0 : LSTM, h = 16, 2 couches, dropout 0, 2 époques ;
- V1 : LSTM, h = 16, 2 couches, dropout 0, 7 époques.

Réentraînement sur 2006–2019 (purgé avant 2020), 5 graines, puis évaluation unique sur 2020–2025. Les colonnes `val_*` contiennent ici les métriques du test.

In [15]:
FINAL_MODELS = {
    "V0": ("v0", {"model": ["lstm"], "hidden": [16], "layers": [2], "dropout": [0.0]}, 2),
    "V1": ("v1", {"model": ["lstm"], "hidden": [16], "layers": [2], "dropout": [0.0]}, 7),
}

finals = []
for name, (arch, cfg, epochs) in FINAL_MODELS.items():
    for seed in range(5):
        finals.append(grid_search(cfg, *FINAL, seed=seed, arch=arch, fixed_epochs=epochs).assign(modèle=name))
finals = pd.concat(finals)
finals.to_csv("models/lstm/results/test_final_v0_v1.csv", index=False)

display(finals.groupby("modèle")[[f"val_{c}" for c in COLS]].agg(["mean", "std"]).round(3))
display(ref[ref["période"] == "test"].set_index("stratégie")[COLS].round(3))

[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.788 | Sharpe +0.80 | 2 époques | 2 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.791 | Sharpe +0.81 | 2 époques | 2 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.811 | Sharpe +0.78 | 2 époques | 2 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.787 | Sharpe +0.79 | 2 époques | 2 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.762 | Sharpe +0.80 | 2 époques | 2 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.747 | Sharpe +0.83 | 7 époques | 42 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.785 | Sharpe +0.81 | 7 époques | 38 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.762 | Sharpe +0.83 | 7 époques | 41 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → loss 2.738 | Sharpe +0.8

val_ce        val_sharpe        val_sharpe_std        val_turnover  \
         mean    std       mean    std           mean    std         mean   
modèle                                                                      
V0      2.788  0.017      0.794  0.012          0.281  0.007        0.022   
V1      2.766  0.026      0.827  0.015          0.285  0.017        0.247   

              val_n_eff         
          std      mean    std  
modèle                          
V0      0.005    14.340  0.518  
V1      0.056    14.302  0.188

,ce,sharpe,sharpe_std,turnover,n_eff
stratégie,,,,,
Oracle,1.225,4.931,0.835,1.593,3.371
Moyenne des cibles du train,2.791,0.775,0.282,0.000,13.419
Équipondéré,2.921,0.845,0.280,0.000,15.000
Inverse volatilité,2.925,0.804,0.284,0.154,13.908


**V2 sur le test.** V2 a été ajoutée après l'évaluation de V0 et V1 ; elle est choisie avec la même règle (meilleure perte moyenne en validation croisée), figée avant ce test, et évaluée une seule fois avec la même procédure (2006–2019, nombre d'époques médian, 5 graines). Tableau final : V0, V1, V2 et les références sur le test.

In [26]:
best_v2 = summary_v2.iloc[0]                                             # meilleure perte moyenne en validation croisée
cfg_v2 = {k: [type(GRID_V2[k][0])(best_v2[k])] for k in GRID_V2}
print("V2 retenu :", {k: v[0] for k, v in cfg_v2.items()}, "|", int(best_v2["epochs"]), "époques")

final_v2 = pd.concat([grid_search(cfg_v2, *FINAL, seed=s, arch="v2", fixed_epochs=int(best_v2["epochs"])).assign(modèle="V2")
                      for s in range(5)])
final_all = pd.concat([pd.read_csv("models/lstm/results/test_final_v0_v1.csv"), final_v2])
final_all = final_all.rename(columns=lambda c: c.replace("val_", "test_"))          # métriques calculées sur le test
final_all.to_csv("models/lstm/results/test_final_v0_v1_v2.csv", index=False)

display(final_all.groupby("modèle")[[f"test_{c}" for c in COLS]].agg(["mean", "std"]).round(3))
display(ref[ref["période"] == "test"].set_index("stratégie")[COLS].round(3))

V2 retenu : {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} | 13 époques
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.709 | Sharpe +0.86 | 13 époques | 51 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.707 | Sharpe +0.86 | 13 époques | 52 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.759 | Sharpe +0.83 | 13 époques | 51 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.772 | Sharpe +0.83 | 13 époques | 50 s
[1/1] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → loss 2.802 | Sharpe +0.77 | 13 époques | 53 s


test_ce        test_sharpe        test_sharpe_std        test_turnover  \
          mean    std        mean    std            mean    std          mean   
modèle                                                                          
V0       2.788  0.017       0.794  0.012           0.281  0.007         0.022   
V1       2.766  0.026       0.827  0.015           0.285  0.017         0.247   
V2       2.750  0.041       0.829  0.039           0.265  0.015         0.334   

              test_n_eff         
          std       mean    std  
modèle                           
V0      0.005     14.340  0.518  
V1      0.056     14.302  0.188  
V2      0.094     12.552  0.615

,ce,sharpe,sharpe_std,turnover,n_eff
stratégie,,,,,
Oracle,1.225,4.931,0.835,1.593,3.371
Moyenne des cibles du train,2.791,0.775,0.282,0.000,13.419
Équipondéré,2.921,0.845,0.280,0.000,15.000
Inverse volatilité,2.925,0.804,0.284,0.154,13.908


## 9. Annexe : premiers essais sur le découpage simple

Essais réalisés avant la mise en place du protocole : grid search sur le seul découpage train (2006–2016) / validation (2017–2019), puis entraînement détaillé de la configuration retenue pour V0 avec ses courbes d'apprentissage. Ils ne servent pas à la sélection finale.

In [ ]:
GRID_HOLDOUT = {
    "model":   ["lstm", "gru"],
    "hidden":  [8, 16, 32],     # taille de l'état caché
    "layers":  [1, 2],          # nombre de couches
    "dropout": [0.0, 0.2],
    "L":       [20, 60, 120],   # profondeur historique (jours)
}
res_v0 = grid_search(GRID_HOLDOUT, seed=0)
res_v0.to_csv("models/lstm/results/grid_v0_seed0.csv", index=False)
res_v0.round(3).head(15)

[1/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 20} → loss 2.653 | Sharpe +1.20 | 7 époques | 1 s
[2/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 60} → loss 2.653 | Sharpe +1.20 | 7 époques | 2 s
[3/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 120} → loss 2.653 | Sharpe +1.20 | 7 époques | 2 s
[4/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 20} → loss 2.653 | Sharpe +1.19 | 7 époques | 1 s
[5/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 60} → loss 2.653 | Sharpe +1.19 | 7 époques | 2 s
[6/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 120} → loss 2.653 | Sharpe +1.19 | 7 époques | 2 s
[7/72] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0, 'L': 20} → loss 2.654 | Sharpe +1.19 | 5 époques | 1 s
[8/72] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0, 'L': 60} → loss 2.654 | Sharpe +1.19 | 5 époques | 2 s
[9/72] {'model': 'lstm

In [ ]:
lstm_V0 = V0(hidden=8,layers=1,dropout=0.2)

L, SEED = 60, 0
LR, WD, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 1e-4, 128, 100, 3

tr_i, va_i = idx["train"], idx["val"]

S = np.stack([logr[i - L + 1:i + 1] for i in pos]).astype(np.float32)
S /= S[tr_i].std()
Yt = torch.tensor(Y.to_numpy(), dtype=torch.float32)
x_tr, x_va = torch.tensor(S[tr_i]), torch.tensor(S[va_i]).to(DEVICE)
y_tr, y_va = Yt[tr_i], Yt[va_i].to(DEVICE)

criterion = lambda p, y: -(y * torch.log(p + 1e-8)).sum(-1).mean()    # entropie croisée

torch.manual_seed(SEED)
lstm_V0 = V0(hidden=8, layers=1, dropout=0.2).to(DEVICE)
opt = torch.optim.Adam(lstm_V0.parameters(), lr=LR, weight_decay=WD)

best, best_ep, best_state, wait = float("inf"), 0, None, 0
history = {"train": [], "val": []}
for ep in range(MAX_EPOCHS):
    lstm_V0.train()
    perm = torch.randperm(len(tr_i))
    for b in range(0, len(perm), BATCH):
        j = perm[b:b + BATCH]
        loss = criterion(lstm_V0(x_tr[j].to(DEVICE)), y_tr[j].to(DEVICE))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_V0.parameters(), 1.0)
        opt.step()

    lstm_V0.eval()
    with torch.no_grad():
        tr_loss = criterion(lstm_V0(x_tr.to(DEVICE)), y_tr.to(DEVICE)).item()
        val_loss = criterion(lstm_V0(x_va), y_va).item()
    history["train"].append(tr_loss); history["val"].append(val_loss)
    print(f"époque {ep + 1:3d} | train {tr_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best - 1e-4:                   
        best, best_ep, wait = val_loss, ep + 1, 0
        best_state = {k: t.clone() for k, t in lstm_V0.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            break

lstm_V0.load_state_dict(best_state)               # on garde les poids de la meilleure époque
print(f"\nmeilleure époque : {best_ep} | CE val {best:.3f} (référence : 2.642)")

# poids prédits sur toute la période + Sharpe (même fonction que les repères)
lstm_V0.eval()
with torch.no_grad():
    W_v0 = lstm_V0(torch.tensor(S).to(DEVICE)).cpu().numpy()    # (T, 16)
print("Sharpe val :", round(metrics(W_v0[va_i], va_i)["sharpe"], 2), "(équipondéré : +1.04)")